# TearSheet.ipynb

- doesn't actually generate a tearsheet like a classic print [S&P or Value Line tearsheet](https://financetrain.com/what-is-a-tear-sheet) on a company
- asks Perplexity a few basic questions about the company
- gets info from AlphaVantage
- opens a bunch of browser tabs from major services about the company
- could potentially save those and scrape them to make a tearsheet.


Updated version
- profile - get info from perplexity , other company profiles, openbb, sec filings, wikipedia generate a description of the company, focus on business model history of the company
- news
  - get latest news
  - scrape news using scraper - grab headlines matching pattern
  - filter stuff by how relevant to the company
- chart
- key ratios, top holders
- generate tear sheet using tools
- question answering chatbot using tools


In [3]:
import dotenv
import sys
import os
import re
from datetime import datetime, timedelta
import time
from typing import List, Optional, Dict, Any
from urllib.parse import urljoin, urlparse
from pathlib import Path

import numpy as np
# should require numpy < 2 for pandas_ta or uses more recent ta-lib module
np.NaN = np.nan

import openbb
from openbb import obb
from openbb_core.app.model.obbject import OBBject

import pandas as pd
# import pandas_ta as ta

import requests
from bs4 import BeautifulSoup
import html2text

import json
import aiohttp
import tempfile

import openai
from openai import OpenAI

import IPython
from IPython.display import HTML, Image, Markdown, display

from selenium import webdriver
from selenium.webdriver.common.by import By
# use firefox because it updates less often, can disable updates
# recommend importing profile from Chrome for cookies, passwords
# looks less like a bot with more user cruft in the profile
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.firefox.service import Service

import wikipedia

import langchain
from langchain_openai import ChatOpenAI
from langchain.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.prompts import ChatPromptTemplate
from langchain.schema import HumanMessage

# from newsapi import NewsApiClient
# # %pip install mem0ai
# import mem0
# from mem0 import MemoryClient

# brew install ta-lib
# https://ta-lib.org/

In [4]:
dotenv.load_dotenv()


True

In [5]:
# Get current year
current_year = datetime.now().year
print(f"Current year: {current_year}")

last_year = datetime.now().year - 1
print(f"Last year: {last_year}")

# Get date 1 year ago (approximate - 365 days)
one_year_ago = datetime.now() - timedelta(days=365)
print(f"Date 1 year ago: {one_year_ago.strftime('%Y-%m-%d')}")

date_today = datetime.now() 
print(f"Date now: {date_today.strftime('%Y-%m-%d')}")


Current year: 2025
Last year: 2024
Date 1 year ago: 2024-08-05
Date now: 2025-08-05


In [6]:
company = "Tesla"
symbol = "TSLA"



In [ ]:
temp_dir = tempfile.mkdtemp(prefix='t', dir='tmp')
temp_dir


In [ ]:
temp_dir = 'tx0ybr2zo'

In [ ]:
def replace_citations(text, citations):
    def replace_match(match):
        citation_num = int(match.group(1))
        if citation_num <= len(citations):
            url = citations[citation_num - 1]  # Citations are 0-indexed, references are 1-indexed
            return f" [{citation_num}]({url})"
        return match.group(0)  # Return original if citation not found

    return re.sub(r'\[(\d+)\]', replace_match, text)

def fetch_perplexity(system_prompt, user_prompt, return_images=False):
    perplexity_url = "https://api.perplexity.ai/chat/completions"
    perplexity_model = "sonar-pro"

    payload = {
        "model": perplexity_model,
        "messages": [
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": user_prompt,
            }
        ],
    "return_citations": True,          # Enable citations for markdown links
    "return_images": return_images,    # Optional: disable images if not needed
    "return_related_questions": False  # Optional: disable related questions
}

    perplexity_headers = {
        "Authorization": f"Bearer {os.getenv('PERPLEXITY_API_KEY')}",
        "accept": "application/json",
        "content-type": "application/json"
    }

    response = requests.post(perplexity_url, json=payload, headers=perplexity_headers)
    response_data = response.json()

    response_str = response_data['choices'][0]['message']['content']
    response_str = response_str.replace("$", r"\$")
    citations = response_data.get('citations', [])
    response_str = replace_citations(response_str, citations)
    if return_images:
        images = response_data.get('images', [])
        if images:
            image_str = f"![Image 1]({images.pop(0)['image_url']})\n\n"
            response_str = image_str + response_str
    return response_str


In [ ]:
def get_perplexity_profile(company, symbol, return_images=False):

    system_prompt = """
You will act as a securities analyst and investment advisor with deep knowledge of financial markets,
securities analysis, portfolio management. You will maintain a professional yet engaging tone,
in the style of a senior investment bank research analyst.
"""

    user_prompt = f"""You will focus on {company} ({symbol}), and provide a comprehensive analysis covering the following aspects:

Company Profile: An overview of {company}, including its lines of business, history, and recent key developments.

Major News: Significant events related to {company} or its industry impacting its stock.

Financial Performance: Recent earnings reports and stock performance compared to expectations, changes in dividends or stock buybacks.

Analyst Coverage: summarize recent changes to analysts' ratings noting which analyst and firms made upgrades or downgrades; summarize any recent short seller reports noting the firm and analyst.

Product Announcements: Launch of new products, strategic initiatives, or restructurings.

Strategic Moves: Information on deals, partnerships, mergers, acquisitions, divestitures, joint ventures, and major new business and revenue.

Securities Offerings: Announcements related to stock or bond issuances, buybacks, special dividends, or stock splits.

Management Changes: Significant personnel changes within {company}.

Stock Price Movements: Notable stock price changes and their reasons.

Timeline and Future Outlook: A timeline of these events """

    return fetch_perplexity(system_prompt, user_prompt, return_images=return_images)


In [ ]:
perplexity_str = get_perplexity_profilee(company, symbol, return_images=True)
with open(f'{temp_dir}/{symbol}_perplexity_profile.md', 'w', encoding='utf-8') as f:
    f.write(perplexity_str)

display(Markdown(perplexity_str))


In [ ]:
def get_perplexity_analyst_ratings(company, symbol, return_images=False):

    system_prompt = """
You will act as a securities analyst and investment advisor with deep knowledge of financial markets,
securities analysis, portfolio management. You will maintain a professional yet engaging tone,
in the style of a senior investment bank research analyst.
"""

    user_prompt = f"""What are the current analyst ratings on {company} ({symbol})?
    Which short sellers issued reports on {company} since {last_year}, if any? Summarize any notable analyst reports."""

    return fetch_perplexity(system_prompt, user_prompt, return_images=False)

perplexity_str = get_perplexity_analyst_ratings(company, symbol)
with open(f'{temp_dir}/{symbol}_perplexity_ratings.md', 'w', encoding='utf-8') as f:
    f.write(perplexity_str)
display(Markdown(perplexity_str))


In [ ]:
def get_perplexity_news(company, symbol, return_images=False):

    system_prompt = """
You will act as a securities analyst and investment advisor with deep knowledge of financial markets,
securities analysis, portfolio management. You will maintain a professional yet engaging tone,
in the style of a senior investment bank research analyst.
"""

    user_prompt = f"""What are there most impactful news stories about {company} ({symbol}) since {last_year}, including management profiles and investigative reports?
For any notable stories, provide the date of publication and the media that reported them."""

    return fetch_perplexity(system_prompt, user_prompt, return_images=False)

perplexity_str = get_perplexity_news(company, symbol)
with open(f'{temp_dir}/{symbol}_perplexity_news.md', 'w', encoding='utf-8') as f:
    f.write(perplexity_str)
display(Markdown(perplexity_str))


In [ ]:
company_list = wikipedia.search(company)
company_list


In [ ]:
def get_best_wikipedia_match(company_description: str, wikipedia_matches: List[str]) -> str:
    """
    Use LangChain with ChatOpenAI to find the best Wikipedia page match for a company.
    
    Args:
        company_description (str): Description of the company (e.g., "Tesla (symbol TSLA)")
        wikipedia_matches (List[str]): List of Wikipedia page titles
    
    Returns:
        str: The best matching Wikipedia page title
    """
    
    # Initialize the Chat LLM
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
    
    # Create the chat prompt template
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant that selects the most appropriate Wikipedia page for a given company from a list of options."),
        ("human", """I am looking for the wikipedia page of {company_description}.

From the list of wikipedia articles below, tell me the title of the page that is most likely the best matching wikipedia page for {company_description}.

Wikipedia articles:
{wikipedia_matches}

Return exactly one page title from this list without any modification. Do not add quotes, explanations, or any other text - just return the exact title as it appears in the list.

Requirements:
- Must be exactly one of the titles from the provided list
- No modifications, quotes, or additional text
- Focus on the main company page, not specific products or history pages

Best match:""")
    ])
    
    # Format the matches as a bulleted list
    formatted_matches = "\n".join([f"• {match}" for match in wikipedia_matches])
    
    # Create and invoke the chain
    chain = prompt | llm
    
    try:
        result = chain.invoke({
            "company_description": company_description,
            "wikipedia_matches": formatted_matches
        })
        
        # Extract the content from the AIMessage
        selected_match = result.content.strip()
        
        # Validate that the result is actually in the original list
        if selected_match in wikipedia_matches:
            return selected_match
        else:
            # Fallback: try to find a close match or return the first corporate-looking entry
            for match in wikipedia_matches:
                if "Inc." in match or "Corp." in match or (company_description.split()[0].lower() in match.lower() and len(match.split()) <= 3):
                    return match
            return wikipedia_matches[0]  # Last resort
            
    except Exception as e:
        print(f"Error occurred: {e}")
        # Simple fallback logic
        for match in wikipedia_matches:
            if "Inc." in match:
                return match
        return wikipedia_matches[0]
    
company_query = f"{company} (symbol {symbol})"
best_match = get_best_wikipedia_match(company_query, company_list)

print(f"Best Wikipedia match for '{company_query}': {best_match}")




In [ ]:
page_object = wikipedia.page(title=best_match, auto_suggest=False)

# printing title
print(page_object.original_title)

# printing links on that page object
#print(page_object.links[0:10])

# printing html of page_object
h = html2text.HTML2Text()
h.ignore_images = True
h.body_width = 0
h.unicode_snob = True
h.baseurl = 'https://en.wikipedia.org'
markdown_content = h.handle(page_object.html())
with open(f'{temp_dir}/{symbol}_wikipedia.md', 'w', encoding='utf-8') as f:
    f.write(perplexity_str)
display(Markdown(markdown_content))



In [ ]:
os.getenv("SEC_USER")


In [ ]:
# free API for edgar filing
import sec_parser as sp
from sec_downloader import Downloader

def fn_get_10k_item_from_symbol(symbol, item="1"):
    """
    Get item 1 (or other number) of the latest 10-K annual report filing for a given symbol.

    Args:
        symbol (str): The symbol of the equity.
        item (str):   The item number to return.

    Returns:
        str: The item requested from the latest 10-K annual report filing, or None if not found.

    """

    item_text = ""
    try:
        print("Getting 10-K Item 1 for %s" % symbol)
        dl = Downloader(os.getenv("SEC_FIRM"), os.getenv("SEC_USER"))
        html = dl.get_filing_html(ticker=symbol, form="10-K")
        print("HTML length: %d characters" % len(html))
        elements = sp.Edgar10QParser().parse(html)
        tree = sp.TreeBuilder().build(elements)
        # look for e.g. "Item 1."
        # sections = [n for n in tree.nodes if re.match(r"^ITEM 1[A-Z]?\.", n.text.strip().upper())]
        sections = [n for n in tree.nodes if re.match(
            r"^ITEM\s+" + item, n.text.strip().upper())]
        print("Sections: %d" % len(sections))
        if len(sections) == 0:
            return ""
        item_node = sections[0]
        item_text = item_node.text + "\n\n" + \
            "\n".join([n.text for n in item_node.get_descendants()])
        print("Item text: %d characters" % len(item_text))
    except Exception as e:
        print("Error getting 10-K item: %s" % e)
    return item_text

edgar_10k_item1 = fn_get_10k_item_from_symbol(symbol)
with open(f'{temp_dir}/{symbol}_edgar_10k_item1.txt', 'w', encoding='utf-8') as f:
    f.write(edgar_10k_item1)
print(edgar_10k_item1)


In [7]:

obb.account.login(email=os.environ['OPENBB_USER'], password=os.environ['OPENBB_PW'], remember_me=True)


In [8]:
obb.account.login(email=os.environ['OPENBB_USER'], password=os.environ['OPENBB_PW'], remember_me=True)
obj = obb.equity.compare.peers(symbol=symbol, provider='fmp')
peers = obj.results
peers


FMPEquityPeersData(peers_list=['AMZN', 'BYDDF', 'BYDDY', 'F', 'GM', 'LCID', 'MULN', 'NIO', 'RIVN', 'TM'])

'{"peers_list": ["AMZN", "BYDDF", "BYDDY", "F", "GM", "LCID", "MULN", "NIO", "RIVN", "TM"]}'

In [ ]:
for peer in peers.peers_list:
    try:
        retval = []
        obj = obb.equity.profile(peer)
        results = obj.results[0]
        desc = ""
        if results.short_description:
            desc = results.short_description
        elif results.long_description:
            desc = results.long_description

        retstr = f"""
Symbol:        {peer}
Name:          {results.name}
Country:       {results.hq_country}
Industry:      {results.industry_category}
Description:   {desc}
"""
#         retval.append(results.stock_exchange)
        print(retstr)
    except Exception as e:
#         print(e)
#         print()
        continue
    print()


In [26]:
import yfinance as yf
import pandas as pd

def get_fundamental_ratios(symbol):
    """
    Get a comprehensive DataFrame of fundamental ratios and statistics
    """
    ticker = yf.Ticker(symbol)
    info = ticker.info
    
    # Organize ratios by category
    valuation_ratios = {
        'Trailing P/E': info.get('trailingPE'),
        'Forward P/E': info.get('forwardPE'),
        'PEG Ratio': info.get('pegRatio'),
        'Price/Sales (ttm)': info.get('priceToSalesTrailing12Months'),
        'Price/Book': info.get('priceToBook'),
        'Enterprise Value/Revenue': info.get('enterpriseToRevenue'),
        'Enterprise Value/EBITDA': info.get('enterpriseToEbitda'),
    }
    
    financial_highlights = {
        'Market Cap': info.get('marketCap'),
        'Enterprise Value': info.get('enterpriseValue'),
        'Revenue (ttm)': info.get('totalRevenue'),
        'Gross Profit (ttm)': info.get('grossProfits'),
        'EBITDA': info.get('ebitda'),
        'Net Income (ttm)': info.get('netIncomeToCommon'),
    }
    
    profitability_ratios = {
        'Profit Margin': info.get('profitMargins'),
        'Operating Margin': info.get('operatingMargins'),
        'Gross Margin': info.get('grossMargins'),
        'EBITDA Margin': info.get('ebitdaMargins'),
        'Return on Assets': info.get('returnOnAssets'),
        'Return on Equity': info.get('returnOnEquity'),
    }
    
    liquidity_ratios = {
        'Current Ratio': info.get('currentRatio'),
        'Quick Ratio': info.get('quickRatio'),
        'Total Cash': info.get('totalCash'),
        'Total Debt': info.get('totalDebt'),
        'Debt/Equity': info.get('debtToEquity'),
    }
    
    per_share_data = {
        'Earnings Per Share (ttm)': info.get('trailingEps'),
        'Book Value Per Share': info.get('bookValue'),
        'Revenue Per Share': info.get('revenuePerShare'),
        'Operating Cash Flow Per Share': info.get('operatingCashflow', 0) / info.get('sharesOutstanding', 1) if info.get('sharesOutstanding') else None,
    }
    
    # Combine all categories
    all_ratios = {
        **valuation_ratios,
        **financial_highlights, 
        **profitability_ratios,
        **liquidity_ratios,
        **per_share_data
    }
    
    # Create DataFrame
    df = pd.DataFrame(list(all_ratios.items()), columns=['Metric', symbol])
    df['Category'] = (
        ['Valuation'] * len(valuation_ratios) +
        ['Financial Highlights'] * len(financial_highlights) +
        ['Profitability'] * len(profitability_ratios) +
        ['Liquidity'] * len(liquidity_ratios) +
        ['Per Share'] * len(per_share_data)
    )

    return df[['Category', 'Metric', symbol]]

# Usage
ratios_df = get_fundamental_ratios('TSLA')
display(Markdown(ratios_df.to_markdown()))


|    | Category             | Metric                        |          TSLA |
|---:|:---------------------|:------------------------------|--------------:|
|  0 | Valuation            | Trailing P/E                  | 184.862       |
|  1 | Valuation            | Forward P/E                   |  95.284       |
|  2 | Valuation            | PEG Ratio                     | nan           |
|  3 | Valuation            | Price/Sales (ttm)             |  10.7394      |
|  4 | Valuation            | Price/Book                    |  12.8735      |
|  5 | Valuation            | Enterprise Value/Revenue      |  10.511       |
|  6 | Valuation            | Enterprise Value/EBITDA       |  85.899       |
|  7 | Financial Highlights | Market Cap                    |   9.95761e+11 |
|  8 | Financial Highlights | Enterprise Value              |   9.74612e+11 |
|  9 | Financial Highlights | Revenue (ttm)                 |   9.272e+10   |
| 10 | Financial Highlights | Gross Profit (ttm)            |   1.6207e+10  |
| 11 | Financial Highlights | EBITDA                        |   1.1346e+10  |
| 12 | Financial Highlights | Net Income (ttm)              |   5.879e+09   |
| 13 | Profitability        | Profit Margin                 |   0.06344     |
| 14 | Profitability        | Operating Margin              |   0.04103     |
| 15 | Profitability        | Gross Margin                  |   0.1748      |
| 16 | Profitability        | EBITDA Margin                 |   0.12237     |
| 17 | Profitability        | Return on Assets              |   0.02911     |
| 18 | Profitability        | Return on Equity              |   0.08177     |
| 19 | Liquidity            | Current Ratio                 |   2.037       |
| 20 | Liquidity            | Quick Ratio                   |   1.359       |
| 21 | Liquidity            | Total Cash                    |   3.6782e+10  |
| 22 | Liquidity            | Total Debt                    |   1.3134e+10  |
| 23 | Liquidity            | Debt/Equity                   |  16.823       |
| 24 | Per Share            | Earnings Per Share (ttm)      |   1.67        |
| 25 | Per Share            | Book Value Per Share          |  23.981       |
| 26 | Per Share            | Revenue Per Share             |  28.862       |
| 27 | Per Share            | Operating Cash Flow Per Share |   4.88769     |

In [40]:

def gfr(symbol):
    "helper function to create ratios dataframe"
    ticker = yf.Ticker(symbol)
    info = ticker.info

    # Organize ratios by category
    valuation_ratios = {
        'Trailing P/E': info.get('trailingPE'),
        'Forward P/E': info.get('forwardPE'),
        'PEG Ratio': info.get('pegRatio'),
        'Price/Sales (ttm)': info.get('priceToSalesTrailing12Months'),
        'Price/Book': info.get('priceToBook'),
        'Enterprise Value/Revenue': info.get('enterpriseToRevenue'),
        'Enterprise Value/EBITDA': info.get('enterpriseToEbitda'),
    }

    financial_highlights = {
        'Market Cap': info.get('marketCap'),
        'Enterprise Value': info.get('enterpriseValue'),
        'Revenue (ttm)': info.get('totalRevenue'),
        'Gross Profit (ttm)': info.get('grossProfits'),
        'EBITDA': info.get('ebitda'),
        'Net Income (ttm)': info.get('netIncomeToCommon'),
    }

    profitability_ratios = {
        'Profit Margin': info.get('profitMargins'),
        'Operating Margin': info.get('operatingMargins'),
        'Gross Margin': info.get('grossMargins'),
        'EBITDA Margin': info.get('ebitdaMargins'),
        'Return on Assets': info.get('returnOnAssets'),
        'Return on Equity': info.get('returnOnEquity'),
    }

    liquidity_ratios = {
        'Current Ratio': info.get('currentRatio'),
        'Quick Ratio': info.get('quickRatio'),
        'Total Cash': info.get('totalCash'),
        'Total Debt': info.get('totalDebt'),
        'Debt/Equity': info.get('debtToEquity'),
    }

    per_share_data = {
        'Earnings Per Share (ttm)': info.get('trailingEps'),
        'Book Value Per Share': info.get('bookValue'),
        'Revenue Per Share': info.get('revenuePerShare'),
        'Operating Cash Flow Per Share': info.get('operatingCashflow', 0) / info.get('sharesOutstanding', 1) if info.get('sharesOutstanding') else None,
    }

    # Combine all categories
    all_ratios = {
        **valuation_ratios,
        **financial_highlights,
        **profitability_ratios,
        **liquidity_ratios,
        **per_share_data
    }

    # Create DataFrame
    df = pd.DataFrame(list(all_ratios.items()), columns=['Metric', symbol])
    df['Category'] = (
        ['Valuation'] * len(valuation_ratios) +
        ['Financial Highlights'] * len(financial_highlights) +
        ['Profitability'] * len(profitability_ratios) +
        ['Liquidity'] * len(liquidity_ratios) +
        ['Per Share'] * len(per_share_data)
    )
    return df[["Category", "Metric", symbol]]


In [44]:
"Policy and regulation".title()

'Policy And Regulation'

In [41]:
gfr('tsla') 


,Category,Metric,tsla
0,Valuation,Trailing P/E,1.848623e+02
1,Valuation,Forward P/E,9.528395e+01
2,Valuation,PEG Ratio,NaN
3,Valuation,Price/Sales (ttm),1.073944e+01
4,Valuation,Price/Book,1.287353e+01
5,Valuation,Enterprise Value/Revenue,1.051100e+01
6,Valuation,Enterprise Value/EBITDA,8.589900e+01
7,Financial Highlights,Market Cap,9.957609e+11
8,Financial Highlights,Enterprise Value,9.746123e+11
9,Financial Highlights,Revenue (ttm),9.272000e+10


In [37]:
symboldf = get_fundamental_ratios(symbol)

obb.account.login(email=os.environ['OPENBB_USER'], password=os.environ['OPENBB_PW'], remember_me=True)
obj = obb.equity.compare.peers(symbol=symbol, provider='fmp')
peers = obj.results
peers_list=peers.peers_list

peers_dflist = [get_fundamental_ratios(p).iloc[:,2] for p in peers_list]
pd.concat([symboldf] + peers_dflist, axis=1)


,Category,Metric,TSLA,AMZN,BYDDF,BYDDY,F,GM,LCID,MULN,NIO,RIVN,TM
0,Valuation,Trailing P/E,1.848623e+02,3.258384e+01,2.000000e+01,1.990141e+01,1.417949e+01,8.027481e+00,NaN,NaN,NaN,NaN,7.441575e+00
1,Valuation,Forward P/E,9.528395e+01,3.475610e+01,1.713596e+00,4.500000e+00,6.320000e+00,4.974456e+00,-2.750000e+00,8.800000e-02,-5.294117e+00,-4.434307e+00,1.150190e+01
2,Valuation,PEG Ratio,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Valuation,Price/Sales (ttm),1.073944e+01,3.402228e+00,1.570778e-01,1.569489e-01,2.376100e-01,2.668458e-01,8.545063e+00,7.787777e-02,1.575678e-01,2.907481e+00,4.924489e-03
4,Valuation,Price/Book,1.287353e+01,6.826674e+00,5.724422e-01,5.696203e-01,9.769455e-01,7.582378e-01,2.318008e+00,NaN,-2.571429e+01,2.237981e+00,6.592602e-02
5,Valuation,Enterprise Value/Revenue,1.051100e+01,3.468000e+00,3.500000e-02,3.500000e-02,9.480000e-01,8.980000e-01,9.152000e+00,3.047000e+00,4.120000e-01,2.382000e+00,6.900000e-01
6,Valuation,Enterprise Value/EBITDA,8.589900e+01,1.736200e+01,2.490000e-01,2.440000e-01,2.053100e+01,1.001900e+01,-3.019000e+00,-1.190000e-01,-1.704000e+00,-4.100000e+00,4.703000e+00
7,Financial Highlights,Market Cap,9.957609e+11,2.279622e+12,1.291994e+11,1.290934e+11,4.401725e+10,5.006026e+10,7.435402e+09,6.955740e+05,1.069218e+10,1.455485e+10,2.365562e+11
8,Financial Highlights,Enterprise Value,9.746123e+11,2.323619e+12,2.897114e+10,2.844296e+10,1.755615e+11,1.684471e+11,7.963166e+09,2.721857e+07,2.794399e+10,1.192613e+10,3.314055e+13
9,Financial Highlights,Revenue (ttm),9.272000e+10,6.700380e+11,8.225185e+11,8.225185e+11,1.852500e+11,1.876000e+11,8.701400e+08,8.931612e+06,6.785765e+10,5.006000e+09,4.803671e+13


list

In [21]:
dflist = [get_fundamental_ratios(p) for p in [symbol] + list(peers.peers_list)]
pd


[                Category                         Metric          TSLA
 0              Valuation                   Trailing P/E  1.848623e+02
 1              Valuation                    Forward P/E  9.528395e+01
 2              Valuation                      PEG Ratio           NaN
 3              Valuation              Price/Sales (ttm)  1.073944e+01
 4              Valuation                     Price/Book  1.287353e+01
 5              Valuation       Enterprise Value/Revenue  1.051100e+01
 6              Valuation        Enterprise Value/EBITDA  8.589900e+01
 7   Financial Highlights                     Market Cap  9.957609e+11
 8   Financial Highlights               Enterprise Value  9.746123e+11
 9   Financial Highlights                  Revenue (ttm)  9.272000e+10
 10  Financial Highlights             Gross Profit (ttm)  1.620700e+10
 11  Financial Highlights                         EBITDA  1.134600e+10
 12  Financial Highlights               Net Income (ttm)  5.879000e+09
 13   

In [22]:
pd.concat(dflist, axis=1) 

,Category,Metric,TSLA,Category,Metric,AMZN,Category,Metric,BYDDF,Category,...,MULN,Category,Metric,NIO,Category,Metric,RIVN,Category,Metric,TM
0,Valuation,Trailing P/E,1.848623e+02,Valuation,Trailing P/E,3.258384e+01,Valuation,Trailing P/E,2.000000e+01,Valuation,...,NaN,Valuation,Trailing P/E,NaN,Valuation,Trailing P/E,NaN,Valuation,Trailing P/E,7.441575e+00
1,Valuation,Forward P/E,9.528395e+01,Valuation,Forward P/E,3.475610e+01,Valuation,Forward P/E,1.713596e+00,Valuation,...,8.800000e-02,Valuation,Forward P/E,-5.294117e+00,Valuation,Forward P/E,-4.434307e+00,Valuation,Forward P/E,1.150190e+01
2,Valuation,PEG Ratio,NaN,Valuation,PEG Ratio,NaN,Valuation,PEG Ratio,NaN,Valuation,...,NaN,Valuation,PEG Ratio,NaN,Valuation,PEG Ratio,NaN,Valuation,PEG Ratio,NaN
3,Valuation,Price/Sales (ttm),1.073944e+01,Valuation,Price/Sales (ttm),3.402228e+00,Valuation,Price/Sales (ttm),1.570778e-01,Valuation,...,7.787777e-02,Valuation,Price/Sales (ttm),1.575678e-01,Valuation,Price/Sales (ttm),2.907481e+00,Valuation,Price/Sales (ttm),4.924489e-03
4,Valuation,Price/Book,1.287353e+01,Valuation,Price/Book,6.826674e+00,Valuation,Price/Book,5.724422e-01,Valuation,...,NaN,Valuation,Price/Book,-2.571429e+01,Valuation,Price/Book,2.237981e+00,Valuation,Price/Book,6.592602e-02
5,Valuation,Enterprise Value/Revenue,1.051100e+01,Valuation,Enterprise Value/Revenue,3.468000e+00,Valuation,Enterprise Value/Revenue,3.500000e-02,Valuation,...,3.047000e+00,Valuation,Enterprise Value/Revenue,4.120000e-01,Valuation,Enterprise Value/Revenue,2.382000e+00,Valuation,Enterprise Value/Revenue,6.900000e-01
6,Valuation,Enterprise Value/EBITDA,8.589900e+01,Valuation,Enterprise Value/EBITDA,1.736200e+01,Valuation,Enterprise Value/EBITDA,2.490000e-01,Valuation,...,-1.190000e-01,Valuation,Enterprise Value/EBITDA,-1.704000e+00,Valuation,Enterprise Value/EBITDA,-4.100000e+00,Valuation,Enterprise Value/EBITDA,4.703000e+00
7,Financial Highlights,Market Cap,9.957609e+11,Financial Highlights,Market Cap,2.279622e+12,Financial Highlights,Market Cap,1.291994e+11,Financial Highlights,...,6.955740e+05,Financial Highlights,Market Cap,1.069218e+10,Financial Highlights,Market Cap,1.455485e+10,Financial Highlights,Market Cap,2.365562e+11
8,Financial Highlights,Enterprise Value,9.746123e+11,Financial Highlights,Enterprise Value,2.323619e+12,Financial Highlights,Enterprise Value,2.897114e+10,Financial Highlights,...,2.721857e+07,Financial Highlights,Enterprise Value,2.794399e+10,Financial Highlights,Enterprise Value,1.192613e+10,Financial Highlights,Enterprise Value,3.314055e+13
9,Financial Highlights,Revenue (ttm),9.272000e+10,Financial Highlights,Revenue (ttm),6.700380e+11,Financial Highlights,Revenue (ttm),8.225185e+11,Financial Highlights,...,8.931612e+06,Financial Highlights,Revenue (ttm),6.785765e+10,Financial Highlights,Revenue (ttm),5.006000e+09,Financial Highlights,Revenue (ttm),4.803671e+13


0     3.258384e+01
1     3.475610e+01
2              NaN
3     3.402228e+00
4     6.826674e+00
5     3.468000e+00
6     1.736200e+01
7     2.279622e+12
8     2.323619e+12
9     6.700380e+11
10    3.323830e+11
11    1.338320e+11
12    7.062300e+10
13    1.054000e-01
14    1.143200e-01
15    4.960700e-01
16    1.997400e-01
17    7.699000e-02
18    2.477000e-01
19    1.024000e+00
20    7.680000e-01
21    9.318000e+10
22    1.595700e+11
23    4.780800e+01
24    6.560000e+00
25    3.131100e+01
26    6.337300e+01
27    1.135848e+01
Name: AMZN, dtype: float64

In [ ]:
obj = obb.equity.fundamental.filings(symbol, form_type='10-K')
r = obj.results[0]
latest_10k_url = r.report_url
latest_10k_url


In [ ]:
client = MemoryClient()

messages = [
    {"role": "user", "content": "What is Item 1 from the latest annual report from Tesla (symbol TSLA)"},
    {"role": "assistant", "content": result['content']},
]
client.add(messages, user_id="tearsheet-TSLA", output_format="v1.1")


In [ ]:
messages = [
    {"role": "user", "content": "Thinking of making a sandwich. What do you recommend?"},
    {"role": "assistant", "content": "How about adding some cheese for extra flavor?"},
    {"role": "user", "content": "Actually, I don't like cheese."},
    {"role": "assistant", "content": "I'll remember that you don't like cheese for future recommendations."}
]
client.add(messages, user_id="alex")


In [ ]:
# Example showing location and preference-aware recommendations
query = "I'm craving some pizza. Any recommendations?"
filters = {
    "AND": [
        {
            "user_id": "alex"
        }
    ]
}
client.search(query, version="v2", filters=filters)

In [ ]:
import os
import pandas as pd
import numpy as np
import aiohttp
import talib
from datetime import datetime, timedelta
from typing import Dict, Any


class MarketData:
    """Handles all market data fetching operations."""

    def __init__(self):
        self.api_key = os.getenv("TIINGO_API_KEY")
        if not self.api_key:
            raise ValueError("TIINGO_API_KEY not found in environment")

        self.headers = {"Content-Type": "application/json", "Authorization": f"Token {self.api_key}"}

    async def get_historical_data(self, symbol: str, lookback_days: int = 365) -> pd.DataFrame:
        """
        Fetch historical daily data for a given symbol.

        Args:
            symbol (str): The stock symbol to fetch data for.
            lookback_days (int): Number of days to look back from today.

        Returns:
            pd.DataFrame: DataFrame containing historical market data.

        Raises:
            ValueError: If the symbol is invalid or no data is returned.
            Exception: For other unexpected issues during the fetch operation.
        """
        end_date = datetime.now()
        start_date = end_date - timedelta(days=lookback_days)

        url = (
            f"https://api.tiingo.com/tiingo/daily/{symbol}/prices?"
            f'startDate={start_date.strftime("%Y-%m-%d")}&'
            f'endDate={end_date.strftime("%Y-%m-%d")}'
        )

        try:
            async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=10)) as session:
                async with session.get(url, headers=self.headers) as response:
                    if response.status == 404:
                        raise ValueError(f"Symbol not found: {symbol}")
                    response.raise_for_status()
                    data = await response.json()

            if not data:
                raise ValueError(f"No data returned for {symbol}")

            df = pd.DataFrame(data)
            df["date"] = pd.to_datetime(df["date"])
            df.set_index("date", inplace=True)

            df[["open", "high", "low", "close"]] = df[["adjOpen", "adjHigh", "adjLow", "adjClose"]].round(2)
            df["volume"] = df["adjVolume"].astype(int)
            df["symbol"] = symbol.upper()

            return df

        except aiohttp.ClientError as e:
            raise ConnectionError(f"Network error while fetching data for {symbol}: {e}")
        except ValueError as ve:
            raise ve  # Propagate value errors (symbol issues, no data, etc.)
        except Exception as e:
            raise Exception(f"Unexpected error fetching data for {symbol}: {e}")


class TechnicalAnalysis:
    """Technical analysis toolkit using TA-Lib for improved performance."""

    @staticmethod
    def add_core_indicators(df: pd.DataFrame) -> pd.DataFrame:
        """Add a core set of technical indicators using TA-Lib."""
        try:
            # Convert to numpy arrays for TA-Lib (required format)
            high = df["high"].values
            low = df["low"].values
            close = df["close"].values
            volume = df["volume"].values

            # Adding trend indicators (Simple Moving Averages)
            df["sma_20"] = talib.SMA(close, timeperiod=20)
            df["sma_50"] = talib.SMA(close, timeperiod=50)
            df["sma_200"] = talib.SMA(close, timeperiod=200)

            # Adding volatility indicators
            df["atr"] = talib.ATR(high, low, close, timeperiod=14)

            # Calculate Average Daily Range Percentage manually
            daily_range = df["high"] - df["low"]
            adr = daily_range.rolling(window=20).mean()
            df["adrp"] = (adr / df["close"]) * 100

            # Average volume (20-day)
            df["avg_20d_vol"] = df["volume"].rolling(window=20).mean()

            # Adding momentum indicators
            df["rsi"] = talib.RSI(close, timeperiod=14)

            # MACD indicator
            macd, macd_signal, macd_hist = talib.MACD(close, fastperiod=12, slowperiod=26, signalperiod=9)
            df["macd"] = macd
            df["macd_signal"] = macd_signal
            df["macd_histogram"] = macd_hist

            return df

        except KeyError as e:
            raise KeyError(f"Missing column in input DataFrame: {str(e)}")
        except Exception as e:
            raise Exception(f"Error calculating indicators: {str(e)}")

    @staticmethod
    def check_trend_status(df: pd.DataFrame) -> Dict[str, Any]:
        """Analyze the current trend status."""
        if df.empty:
            raise ValueError("DataFrame is empty. Ensure it contains valid data.")

        latest = df.iloc[-1]

        # Handle potential NaN values
        macd_bullish = False
        if not pd.isna(latest["macd"]) and not pd.isna(latest["macd_signal"]):
            macd_bullish = latest["macd"] > latest["macd_signal"]

        return {
            "above_20sma": latest["close"] > latest["sma_20"] if not pd.isna(latest["sma_20"]) else False,
            "above_50sma": latest["close"] > latest["sma_50"] if not pd.isna(latest["sma_50"]) else False,
            "above_200sma": latest["close"] > latest["sma_200"] if not pd.isna(latest["sma_200"]) else False,
            "20_50_bullish": latest["sma_20"] > latest["sma_50"] if not pd.isna(latest["sma_20"]) and not pd.isna(latest["sma_50"]) else False,
            "50_200_bullish": latest["sma_50"] > latest["sma_200"] if not pd.isna(latest["sma_50"]) and not pd.isna(latest["sma_200"]) else False,
            "rsi": latest["rsi"] if not pd.isna(latest["rsi"]) else 0,
            "macd_bullish": macd_bullish,
        }


# Usage example
async def analyze_symbol(symbol: str):
    """Complete analysis workflow for a given symbol."""
    market_data = MarketData()
    tech_analysis = TechnicalAnalysis()

    df = await market_data.get_historical_data(symbol)
    df = tech_analysis.add_core_indicators(df)
    trend = tech_analysis.check_trend_status(df)

    analysis = f"""
Technical Analysis for {symbol}:

Trend Analysis:
- Above 20 SMA: {'✅' if trend['above_20sma'] else '❌'}
- Above 50 SMA: {'✅' if trend['above_50sma'] else '❌'}
- Above 200 SMA: {'✅' if trend['above_200sma'] else '❌'}
- 20/50 SMA Bullish Cross: {'✅' if trend['20_50_bullish'] else '❌'}
- 50/200 SMA Bullish Cross: {'✅' if trend['50_200_bullish'] else '❌'}

Momentum:
- RSI (14): {trend['rsi']:.2f}
- MACD Bullish: {'✅' if trend['macd_bullish'] else '❌'}

Latest Price: ${df['close'].iloc[-1]:.2f}
Average True Range (14): {df['atr'].iloc[-1]:.2f}
Average Daily Range Percentage: {df['adrp'].iloc[-1]:.2f}%
Average Volume (20D): {df['avg_20d_vol'].iloc[-1]:,.0f}
"""
#     print(analysis)
    return df, trend, analysis

# Example usage:
# df, trend = await analyze_symbol("AAPL")

In [ ]:
temp_dir = "tmp/tx0ybr2zo" 


In [ ]:
df, trend, analysis_str = await analyze_symbol(symbol)
with open(f'{temp_dir}/{symbol}_technicals.md', 'w', encoding='utf-8') as f:
    f.write(analysis_str)
print(analysis_str)

In [ ]:
from openai import OpenAI
from mem0 import Memory

openai_client = OpenAI()
memory = Memory()

def chat_with_memories(message: str, user_id: str = "default_user") -> str:
    # Retrieve relevant memories
    relevant_memories = memory.search(query=message, user_id=user_id, limit=3)
    memories_str = "\n".join(f"- {entry['memory']}" for entry in relevant_memories["results"])

    # Generate Assistant response
    system_prompt = f"You are a helpful AI. Answer the question based on query and memories.\nUser Memories:\n{memories_str}"
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": message}]
    response = openai_client.chat.completions.create(model="gpt-4o-mini", messages=messages)
    assistant_response = response.choices[0].message.content

    # Create new memories from the conversation
    messages.append({"role": "assistant", "content": assistant_response})
    memory.add(messages, user_id=user_id)

    return assistant_response

def main():
    print("Chat with AI (type 'exit' to quit)")
    while True:
        user_input = input("You: ").strip()
        if user_input.lower() == 'exit':
            print("Goodbye!")
            break
        print(f"AI: {chat_with_memories(user_input)}")

main()

In [ ]:
filters = {
   "AND": [
      {
         "user_id": "alex"
      }
   ]
}

all_memories = client.get_all(version="v2", filters=filters, page=1, page_size=50)

all_memories


In [ ]:

def extract(url):
    extractor = SECItem1Extractor(url)

    # Analyze document structure first
    print("Document Structure Analysis:")
    structure = extractor.get_document_structure()

    print(f"\nFound {len(structure['anchors'])} anchors:")
    for anchor in structure['anchors'][:10]:  # Show first 10
        print(f"  - {anchor['name']}: {anchor['text']}")

    print(f"\nFound {len(structure['headings'])} item-related headings:")
    for heading in structure['headings']:
        print(f"  - {heading['tag']}: {heading['text']}")

    print(f"\nFound {len(structure['toc_links'])} TOC links:")
    for link in structure['toc_links']:
        print(f"  - {link['text']} -> {link['href']}")

    # Extract Item 1
    print("\n" + "="*50)
    print("EXTRACTING ITEM 1")
    print("="*50)

    result = extractor.extract_item1()
    if result:
        print(f"Successfully extracted using method: {result['method']}")
        print(f"Word count: {result['word_count']}")
        print(f"First 500 characters:\n{result['content'][:500]}...")
    else:
        print("Failed to extract Item 1 content")
    return result


result = extract(latest_10k_url)
print(result['content'])


In [ ]:
print(result['content'])


In [ ]:
# AlphaVantage overview
url = f"https://www.alphavantage.co/query?function=OVERVIEW&symbol={symbol}&apikey={os.environ['ALPHAVANTAGE_API_KEY']}"
r = requests.get(url)
data = r.json()
with open(f'{temp_dir}/{symbol}_fundamintals_av.json', 'w', encoding='utf-8') as f:
    f.write(json.dumps(data))
pd.DataFrame(data, index=[1]).transpose()


In [ ]:
exchange = data['Exchange']

morningstarmap = {'NYSE': 'xnys',
                  'NASDAQ': 'xnas'
                 }


In [ ]:
now = datetime.now()
end_date = now.strftime('%Y%m%dT%H%M')
sdate = now - timedelta(days=2)
# sdate = datetime(now.year, now.month-1, now.day)
start_date = sdate.strftime('%Y%m%dT%H%M')
start_date, end_date

In [ ]:
url = f'https://www.alphavantage.co/query?function=NEWS_SENTIMENT&tickers={symbol}&apikey={os.environ["ALPHAVANTAGE_API_KEY"]}&time_from={start_date}&time_to={end_date}'

r = requests.get(url)
data = r.json()

for item in data['feed']:
    markdown_str = ""
    date_object = datetime.strptime(item['time_published'], "%Y%m%dT%H%M%S")
    display_title = item['title'].replace("$", r"\$")  # so Markdown doesn't interpret as latex escape
    description = item['summary'].replace("$", r"\$")
    markdown_str += f"[{str(date_object)} {display_title}]({item['url']})\n {description}"
    display(Markdown(markdown_str))


In [ ]:
NEWSAPI_API_KEY = os.environ['NEWSAPI_API_KEY']

page_size = 100
q = company
date_24h_ago = datetime.now() - timedelta(hours=24)
formatted_date = date_24h_ago.strftime("%Y-%m-%dT%H:%M:%S")
print(f"Fetching top {page_size} stories matching {q} since {formatted_date} from NewsAPI")

api = NewsApiClient(api_key=NEWSAPI_API_KEY)
# sources = api.get_sources()
# pd.DataFrame(sources['sources'])

baseurl = 'https://newsapi.org/v2/everything'

# Define search parameters
params = {
    'q': q,
    'from': formatted_date,
    'language': 'en',
    'sortBy': 'relevancy',
    'apiKey': NEWSAPI_API_KEY,
    'pageSize': 100
}

# Make API call with headers and params
response = requests.get(baseurl, params=params, timeout=60)
if response.status_code != 200:
    print('ERROR: API call failed.')
    print(response.text)
else:
    data = response.json()
    newsapi_df = pd.DataFrame(data['articles'])

    # Print the articles
    markdown_str = ""
    for row in newsapi_df.itertuples():
#         date_object = datetime.strptime(row.publishedAt, "%Y%m%dT%H%M%S")
        display_title = row.title.replace("$", r"\$")  # so Markdown doesn't interpret as latex escape
        markdown_str += f"[{row.publishedAt} {display_title}]({row.url}) \n\n"

    display(Markdown(markdown_str))


In [ ]:
now = datetime.now()
end_date = now.strftime('%Y-%m-%d')
# start_date = datetime.now() - timedelta(days=30)
sdate = datetime(now.year, now.month-1, now.day)
start_date = sdate.strftime('%Y-%m-%d')
start_date, end_date


In [ ]:
API_KEY = os.environ['NEWSFILTER_API_KEY']
API_ENDPOINT = "https://api.newsfilter.io/search?token={}".format(API_KEY)

# Define the news search parameters
queryString = f"symbols:{symbol} AND publishedAt:[{start_date} TO {end_date}]"

payload = {
    "queryString": queryString,
    "from": 0,
    "size": 10
}

# Send the search query to the Search API
response = requests.post(API_ENDPOINT, json=payload)

# Read the response
articles = response.json()


# Read the response
articles = response.json()

for item in articles['articles']:
    markdown_str = ""
    date_array = item['publishedAt'].split("T")
    datestr = date_array[0] + "T" + date_array[1][:8]
#     print(item['publishedAt'])
#     print(date_array)
#     print(datestr)
    date_object = datetime.strptime(datestr, "%Y-%m-%dT%H:%M:%S")
    display_title = item['title'].replace("$", r"\$")  # so Markdown doesn't interpret as latex escape
    description = item['description'].replace("$", r"\$")

    markdown_str += f"[{str(date_object)} {display_title}]({item['url']})\n {description}"
    display(Markdown(markdown_str))

In [ ]:
# https://sethhobson.com/2025/01/building-a-stock-analysis-server-with-mcp-part-1/

class MarketData:
    """Handles all market data fetching operations."""

    def __init__(self):
        self.api_key = os.getenv("TIINGO_API_KEY")
        if not self.api_key:
            raise ValueError("TIINGO_API_KEY not found in environment")

        self.headers = {"Content-Type": "application/json", "Authorization": f"Token {self.api_key}"}

    async def get_historical_data(self, symbol: str, lookback_days: int = 365) -> pd.DataFrame:
        """
        Fetch historical daily data for a given symbol.

        Args:
            symbol (str): The stock symbol to fetch data for.
            lookback_days (int): Number of days to look back from today.

        Returns:
            pd.DataFrame: DataFrame containing historical market data.

        Raises:
            ValueError: If the symbol is invalid or no data is returned.
            Exception: For other unexpected issues during the fetch operation.
        """
        end_date = datetime.now()
        start_date = end_date - timedelta(days=lookback_days)

        url = (
            f"https://api.tiingo.com/tiingo/daily/{symbol}/prices?"
            f'startDate={start_date.strftime("%Y-%m-%d")}&'
            f'endDate={end_date.strftime("%Y-%m-%d")}'
        )

        try:
            async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=10)) as session:
                async with session.get(url, headers=self.headers) as response:
                    if response.status == 404:
                        raise ValueError(f"Symbol not found: {symbol}")
                    response.raise_for_status()
                    data = await response.json()

            if not data:
                raise ValueError(f"No data returned for {symbol}")

            df = pd.DataFrame(data)
            df["date"] = pd.to_datetime(df["date"])
            df.set_index("date", inplace=True)

            df[["open", "high", "low", "close"]] = df[["adjOpen", "adjHigh", "adjLow", "adjClose"]].round(2)
            df["volume"] = df["adjVolume"].astype(int)
            df["symbol"] = symbol.upper()

            return df

        except aiohttp.ClientError as e:
            raise ConnectionError(f"Network error while fetching data for {symbol}: {e}")
        except ValueError as ve:
            raise ve  # Propagate value errors (symbol issues, no data, etc.)
        except Exception as e:
            raise Exception(f"Unexpected error fetching data for {symbol}: {e}")


In [ ]:
class TechnicalAnalysis:
    """Technical analysis toolkit with improved performance and readability."""

    @staticmethod
    def add_core_indicators(df: pd.DataFrame) -> pd.DataFrame:
        """Add a core set of technical indicators."""
        try:
            # Adding trend indicators
            df["sma_20"] = ta.sma(df["close"], length=20)
            df["sma_50"] = ta.sma(df["close"], length=50)
            df["sma_200"] = ta.sma(df["close"], length=200)

            # Adding volatility indicators and volume
            daily_range = df["high"].sub(df["low"])
            adr = daily_range.rolling(window=20).mean()
            df["adrp"] = adr.div(df["close"]).mul(100)
            df["avg_20d_vol"] = df["volume"].rolling(window=20).mean()

            # Adding momentum indicators
            df["atr"] = ta.atr(df["high"], df["low"], df["close"], length=14)
            df["rsi"] = ta.rsi(df["close"], length=14)
            macd = ta.macd(df["close"], fast=12, slow=26, signal=9)
            if macd is not None:
                df = pd.concat([df, macd], axis=1)

            return df

        except KeyError as e:
            raise KeyError(f"Missing column in input DataFrame: {str(e)}")
        except Exception as e:
            raise Exception(f"Error calculating indicators: {str(e)}")

    @staticmethod
    def check_trend_status(df: pd.DataFrame) -> Dict[str, Any]:
        """Analyze the current trend status."""
        if df.empty:
            raise ValueError("DataFrame is empty. Ensure it contains valid data.")

        latest = df.iloc[-1]
        return {
            "above_20sma": latest["close"] > latest["sma_20"],
            "above_50sma": latest["close"] > latest["sma_50"],
            "above_200sma": latest["close"] > latest["sma_200"],
            "20_50_bullish": latest["sma_20"] > latest["sma_50"],
            "50_200_bullish": latest["sma_50"] > latest["sma_200"],
            "rsi": latest["rsi"],
            "macd_bullish": latest.get("MACD_12_26_9", 0) > latest.get("MACDs_12_26_9", 0),
        }

In [ ]:
async def fetch_technical_analysis(symbol):

    market_data = MarketData()
    tech_analysis = TechnicalAnalysis()
    tadf = await market_data.get_historical_data(symbol)
    tadf = tech_analysis.add_core_indicators(tadf)
    trend = tech_analysis.check_trend_status(df)
    analysis = f"""
    Technical Analysis for {symbol}:

    Trend Analysis:
    - Above 20 SMA: {'✅' if trend['above_20sma'] else '❌'}
    - Above 50 SMA: {'✅' if trend['above_50sma'] else '❌'}
    - Above 200 SMA: {'✅' if trend['above_200sma'] else '❌'}
    - 20/50 SMA Bullish Cross: {'✅' if trend['20_50_bullish'] else '❌'}
    - 50/200 SMA Bullish Cross: {'✅' if trend['50_200_bullish'] else '❌'}

    Momentum:
    - RSI (14): {trend['rsi']:.2f}
    - MACD Bullish: {'✅' if trend['macd_bullish'] else '❌'}

    Latest Price: ${df['close'].iloc[-1]:.2f}
    Average True Range (14): {df['atr'].iloc[-1]:.2f}
    Average Daily Range Percentage: {df['adrp'].iloc[-1]:.2f}%
    Average Volume (20D): {df['avg_20d_vol'].iloc[-1]}
    """
    return analysis

print(await fetch_technical_analysis('TSLA'))



In [ ]:

print(analysis)


In [ ]:
# open pages via selenium and firefox
outputdir = "htmldata"

# Print the formatted time
print(datetime.now().strftime('%H:%M:%S'), "Starting", flush=True)

firefox_app_path = '/Applications/Firefox.app'
# Path to your geckodriver
geckodriver_path = '/Users/drucev/webdrivers/geckodriver'
# Set up Firefox options to use your existing profile
# important for some sites that need a login, also a generic profile fingerprint that looks like a bot might get blocked
firefox_profile_path = '/Users/drucev/Library/Application Support/Firefox/Profiles/x6dmpnoo.default-release-1'
options = Options()
options.profile = firefox_profile_path

print(datetime.now().strftime('%H:%M:%S'), "Initialized profile", flush=True)

# Create a Service object with the path
service = Service(geckodriver_path)

print(datetime.now().strftime('%H:%M:%S'), "Initialized service", flush=True)
# Set up the Firefox driver
driver = webdriver.Firefox(service=service, options=options)

print(datetime.now().strftime('%H:%M:%S'), "Initialized webdriver", flush=True)
sleeptime = 10


In [ ]:
# Open a new tab
def open_tab(driver, url):
    driver.execute_script(f"window.open('{url}');")
    # Switch to last one opened - sometimes hangs?
    # driver.switch_to.window(driver.window_handles[-1])
    # print(url)
    # time.sleep(sleeptime)


In [ ]:
url = f"https://stockcharts.com/h-sc/ui?s={symbol}&id=p33407302522&def=Y&listNum=1#"
source = "Stockcharts"
# Open the page
open_tab(driver, url)

# Wait for the page to load
# time.sleep(sleeptime)
display(Markdown(f"[{source}]({url})"))

In [ ]:
url = f'https://www.bloomberg.com/search?query={company}'
source = "Bloomberg"
open_tab(driver, url)
display(Markdown(f"[{source}]({url})"))

In [ ]:
url = f'https://www.reuters.com/site-search/?query={company}&offset=0'
source = "Reuters"
open_tab(driver, url)
display(Markdown(f"[{source}]({url})"))

In [ ]:
url = f'https://finance.yahoo.com/quote/{symbol}/news'
source = "Yahoo quote"
open_tab(driver, url)
display(Markdown(f"[{source}]({url})"))

In [ ]:
url = f'https://finance.yahoo.com/quote/{symbol}/key-statistics?ltr=1'
source = "Yahoo stats"
open_tab(driver, url)
display(Markdown(f"[{source}]({url})"))

In [ ]:
url = f'https://www.ft.com/search?q={company}'
source = "FT"
open_tab(driver, url)
display(Markdown(f"[{source}]({url})"))

In [ ]:
url = f'https://www.marketwatch.com/investing/stock/{symbol}?mod=search_symbol'
source = "Marketwatch"
open_tab(driver, url)
display(Markdown(f"[{source}]({url})"))

In [ ]:
url = f'https://www.barrons.com/market-data/stocks/{symbol}?mod=searchresults_companyquotes&mod=searchbar&search_keywords={company}&search_statement_type=suggested'
source = "Barrons"
open_tab(driver, url)
display(Markdown(f"[{source}]({url})"))

In [ ]:
url = f'https://www.businessinsider.com/answers#{company}'

source = "Insider"
open_tab(driver, url)
display(Markdown(f"[{source}]({url})"))

In [ ]:
url = f'https://www.google.com/finance/quote/{symbol}:{exchange}'
source = "Google Finance"
open_tab(driver, url)
display(Markdown(f"[{source}]({url})"))

In [ ]:
url = f'https://finviz.com/quote.ashx?t={symbol}&p=d'
source = "FinViz"
open_tab(driver, url)
display(Markdown(f"[{source}]({url})"))

In [ ]:
url = f'https://www.reddit.com/search?q={company}&include_over_18=on&sort=relevance&t=all'
source = "Reddit"
open_tab(driver, url)
display(Markdown(f"[{source}]({url})"))

In [ ]:
morn_exch = morningstarmap[exchange]
url = f'https://www.morningstar.com/stocks/{morn_exch}/{symbol}/quote'
source = "Morningstar"
open_tab(driver, url)
display(Markdown(f"[{source}]({url})"))


In [ ]:
url = f'https://whalewisdom.com/stock/{symbol}'
source = "WhaleWisdom"
open_tab(driver, url)
display(Markdown(f"[{source}]({url})"))


In [ ]:
url = f'https://www.gurufocus.com/stock/{symbol}/guru-trades'
source = "GuruFocus"
open_tab(driver, url)
display(Markdown(f"[{source}]({url})"))


In [ ]:
# https://www.barchart.com/stocks/highs-lows/lows?orderBy=highPercent1y&orderDir=asc
#     https://stockcharts.com/h-sc/ui?s=%24DJUSEN&id=p33407302522&def=Y&listNum=1
#         https://stockcharts.com/h-sc/ui?s=%24DJUSDN&id=p33407302522&def=Y&listNum=1

In [ ]:
# driver.close()


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime, timedelta

def create_advanced_stock_chart(symbol, period='2y'):
    """
    Create an advanced stock chart with candlesticks, moving averages, volume, and relative performance

    Parameters:
    symbol (str): Stock symbol (e.g., 'TSLA')
    period (str): Time period ('1y', '2y', '5y', etc.)
    """

    # Fetch stock data
    stock = yf.Ticker(symbol)
    df = stock.history(period=period)

    # Fetch S&P 500 data for relative performance
    spy = yf.Ticker('SPY')
    spy_df = spy.history(period=period)

    # Calculate moving averages
    df['MA_13'] = df['Close'].rolling(window=13).mean()  # ~13 weeks (65 trading days)
    df['MA_52'] = df['Close'].rolling(window=52).mean()  # ~52 weeks (260 trading days)

    # Calculate relative performance vs S&P 500
    # Normalize both to starting point and calculate ratio
    stock_normalized = df['Close'] / df['Close'].iloc[0]
    spy_normalized = spy_df['Close'] / spy_df['Close'].iloc[0]
    relative_performance = stock_normalized / spy_normalized

    # Create subplots
    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.02,
        row_heights=[0.6, 0.2, 0.2],
        subplot_titles=(f'{symbol} Stock Chart', 'Volume', f'{symbol} vs S&P 500 Relative Performance')
    )

    # Add candlestick chart
    fig.add_trace(
        go.Candlestick(
            x=df.index,
            open=df['Open'],
            high=df['High'],
            low=df['Low'],
            close=df['Close'],
            name=symbol,
            showlegend=False
        ),
        row=1, col=1
    )

    # Add 13-week moving average
    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=df['MA_13'],
            mode='lines',
            name='13-Week MA',
            line=dict(color='blue', width=1.5),
            opacity=0.8
        ),
        row=1, col=1
    )

    # Add 52-week moving average
    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=df['MA_52'],
            mode='lines',
            name='52-Week MA',
            line=dict(color='red', width=1.5),
            opacity=0.8
        ),
        row=1, col=1
    )

    # Add volume bars
    colors = ['red' if close < open else 'green' for close, open in zip(df['Close'], df['Open'])]
    fig.add_trace(
        go.Bar(
            x=df.index,
            y=df['Volume'],
            name='Volume',
            marker_color=colors,
            opacity=0.6,
            showlegend=False
        ),
        row=2, col=1
    )

    # Add relative performance
    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=relative_performance,
            mode='lines',
            name=f'{symbol} vs SPY',
            line=dict(color='black', width=1.5),
            fill='tonexty' if relative_performance.iloc[-1] > 1 else 'tozeroy',
            fillcolor='rgba(0,255,0,0.1)' if relative_performance.iloc[-1] > 1 else 'rgba(255,0,0,0.1)',
            showlegend=False
        ),
        row=3, col=1
    )

    # Add horizontal line at 1.0 for relative performance
    fig.add_hline(y=1.0, line_dash="dash", line_color="gray", opacity=0.5, row=3, col=1)

    # Update layout
    fig.update_layout(
        title=f'{symbol} - Advanced Stock Analysis',
        height=800,
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        ),
        margin=dict(l=50, r=50, t=80, b=50),
        plot_bgcolor='rgba(240,240,240,0.5)',
        paper_bgcolor='white'
    )

    # Update x-axes
    fig.update_xaxes(
        rangeslider_visible=False,
        showgrid=True,
        gridwidth=1,
        gridcolor='rgba(128,128,128,0.2)'
    )

    # Update y-axes
    fig.update_yaxes(
        showgrid=True,
        gridwidth=1,
        gridcolor='rgba(128,128,128,0.2)',
        title_text="Price ($)",
        row=1, col=1
    )

    fig.update_yaxes(
        title_text="Volume",
        row=2, col=1
    )

    fig.update_yaxes(
        title_text="Relative Performance",
        row=3, col=1
    )

    # Remove range slider from candlestick
    fig.layout.xaxis.rangeslider.visible = False

    return fig

# Example usage
if __name__ == "__main__":
    # Create chart for Tesla
    fig = create_advanced_stock_chart('TSLA', period='2y')
    fig.show()

    # You can also create charts for other stocks
    # fig = create_advanced_stock_chart('AAPL', period='1y')
    # fig.show()

# Additional customization options:

def add_technical_indicators(fig, df, row=1):
    """Add additional technical indicators to the chart"""

    # Bollinger Bands
    window = 20
    df['BB_Middle'] = df['Close'].rolling(window=window).mean()
    df['BB_Std'] = df['Close'].rolling(window=window).std()
    df['BB_Upper'] = df['BB_Middle'] + (df['BB_Std'] * 2)
    df['BB_Lower'] = df['BB_Middle'] - (df['BB_Std'] * 2)

    # Add Bollinger Bands
    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=df['BB_Upper'],
            mode='lines',
            name='BB Upper',
            line=dict(color='purple', width=1, dash='dot'),
            opacity=0.6
        ),
        row=row, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=df['BB_Lower'],
            mode='lines',
            name='BB Lower',
            line=dict(color='purple', width=1, dash='dot'),
            fill='tonexty',
            fillcolor='rgba(128,0,128,0.1)',
            opacity=0.6
        ),
        row=row, col=1
    )

    return fig

def customize_chart_appearance(fig):
    """Apply custom styling to match the original chart aesthetic"""

    fig.update_layout(
        plot_bgcolor='rgba(245,245,220,0.8)',  # Beige background
        paper_bgcolor='white',
        font=dict(size=10, color='black'),
        title_font_size=14
    )

    # Update candlestick colors
    fig.data[0].increasing.fillcolor = 'rgba(0,150,0,0.8)'
    fig.data[0].increasing.line.color = 'rgba(0,100,0,1)'
    fig.data[0].decreasing.fillcolor = 'rgba(200,0,0,0.8)'
    fig.data[0].decreasing.line.color = 'rgba(150,0,0,1)'

    return fig

# Enhanced example with all features
def create_complete_chart(symbol='TSLA', period='2y'):
    """Create a complete chart with all technical indicators and styling"""

    fig = create_advanced_stock_chart(symbol, period)

    # Get the data again for technical indicators
    stock = yf.Ticker(symbol)
    df = stock.history(period=period)

    # Add technical indicators
    fig = add_technical_indicators(fig, df)

    # Apply custom styling
    fig = customize_chart_appearance(fig)

    return fig

# Usage:
# fig = create_complete_chart('TSLA', '2y')
# fig.show()

In [ ]:
import yfinance as yf
import pandas as pd
import plotly.graph_objs as go
from plotly.subplots import make_subplots

def make_stock_chart(symbol):

    # Download weekly data
    tsla = yf.download(symbol, interval="1wk", period="4y")
    tsla.columns = [col[0] if col[1] == symbol else col[0] for col in tsla.columns]
    spx = yf.download("^GSPC", interval="1wk", period="4y")
    spx.columns = [col[0] if col[1] == '^GSPC' else col[0] for col in spx.columns]

    # Compute moving averages
    tsla['MA13'] = tsla['Close'].rolling(window=13).mean()
    tsla['MA52'] = tsla['Close'].rolling(window=52).mean()

    # Compute relative strength vs SPX
    relative = tsla['Close'] / spx['Close']
    tsla['Rel_SPX'] = relative

    # Create figure with secondary y-axis in the first row
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        row_heights=[0.7, 0.3],
        vertical_spacing=0.05,
        specs=[[{"secondary_y": True}], [{}]],  # row 1 has secondary y-axis
        subplot_titles=[f"{symbol} Price with Moving Averages & Volume", f"{symbol} Relative to S&P 500"]
    )

    # --- Row 1: Price Candlesticks & MAs (primary y-axis) ---
    fig.add_trace(go.Candlestick(
        x=tsla.index,
        open=tsla['Open'],
        high=tsla['High'],
        low=tsla['Low'],
        close=tsla['Close'],
        name=symbol,
        increasing_line_color='black',
        decreasing_line_color='red'
    ), row=1, col=1, secondary_y=False)

    fig.add_trace(go.Scatter(
        x=tsla.index,
        y=tsla['MA13'],
        mode='lines',
        name='13-week MA',
        line=dict(color='blue')
    ), row=1, col=1, secondary_y=False)

    fig.add_trace(go.Scatter(
        x=tsla.index,
        y=tsla['MA52'],
        mode='lines',
        name='52-week MA',
        line=dict(color='orange')
    ), row=1, col=1, secondary_y=False)

    # --- Row 1: Volume on right axis (secondary y-axis) ---
    fig.add_trace(go.Bar(
        x=tsla.index,
        y=tsla['Volume'],
        name='Volume',
        marker_color='rgba(0, 128, 0, 0.4)',
        showlegend=False
    ), row=1, col=1, secondary_y=True)

    # --- Row 2: Relative to SPX ---
    fig.add_trace(go.Scatter(
        x=tsla.index,
        y=tsla['Rel_SPX'],
        name=symbol + ' / SPX',
        mode='lines',
        line=dict(color='black')
    ), row=2, col=1)

    # Layout adjustments
    fig.update_layout(
        title=symbol + ' Weekly Chart with MAs, Volume (Right Axis), and Relative Strength',
        height=800,
        xaxis=dict(rangeslider_visible=False),
        showlegend=True
    )

    fig.update_yaxes(title_text="Price", row=1, col=1, secondary_y=False)
    fig.update_yaxes(title_text="Volume", row=1, col=1, secondary_y=True)
    fig.update_yaxes(title_text=f"{symbol} / SPX", row=2, col=1)

    fig.show()

make_stock_chart('TSLA')

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Create figure with secondary y-axis in the first row
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.7, 0.3],
    vertical_spacing=0.05,
    specs=[[{"secondary_y": True}], [{}]],  # row 1 has secondary y-axis
    subplot_titles=["TSLA Price with Moving Averages & Volume", "TSLA Relative to S&P 500"]
)

# --- Row 1: Price Candlesticks & MAs (primary y-axis) ---
fig.add_trace(go.Candlestick(
    x=tsla.index,
    open=tsla['Open'],
    high=tsla['High'],
    low=tsla['Low'],
    close=tsla['Close'],
    name='TSLA',
    increasing_line_color='black',
    decreasing_line_color='red'
), row=1, col=1, secondary_y=False)

fig.add_trace(go.Scatter(
    x=tsla.index,
    y=tsla['MA13'],
    mode='lines',
    name='13-week MA',
    line=dict(color='blue')
), row=1, col=1, secondary_y=False)

fig.add_trace(go.Scatter(
    x=tsla.index,
    y=tsla['MA52'],
    mode='lines',
    name='52-week MA',
    line=dict(color='orange')
), row=1, col=1, secondary_y=False)

# --- Row 1: Volume on right axis (secondary y-axis) ---
fig.add_trace(go.Bar(
    x=tsla.index,
    y=tsla['Volume'],
    name='Volume',
    marker_color='rgba(0, 128, 0, 0.4)',
    showlegend=False
), row=1, col=1, secondary_y=True)

# --- Row 2: Relative to SPX ---
fig.add_trace(go.Scatter(
    x=tsla.index,
    y=tsla['Rel_SPX'],
    name='TSLA / SPX',
    mode='lines',
    line=dict(color='black')
), row=2, col=1)

# Layout adjustments
fig.update_layout(
    title='TSLA Weekly Chart with MAs, Volume (Right Axis), and Relative Strength',
    height=800,
    xaxis=dict(rangeslider_visible=False),
    showlegend=True
)

fig.update_yaxes(title_text="Price", row=1, col=1, secondary_y=False)
fig.update_yaxes(title_text="Volume", row=1, col=1, secondary_y=True)
fig.update_yaxes(title_text="TSLA / SPX", row=2, col=1)

fig.show()

In [ ]:
tsla.columns = ['_'.join(filter(None, col)).strip() for col in tsla.columns.values]
tsla


In [ ]:
tsla = yf.download("TSLA", interval="1wk", period="6y")
tsla.columns = [col[0] if col[1] == 'TSLA' else col[0] for col in tsla.columns]
tsla


In [ ]:
tod

In [ ]:
xxx

In [ ]:
system_prompt = f""" 
You are an expert investment research analyst tasked with creating comprehensive up-to-date company profiles from an institutional investment perspective. Your role is to conduct thorough, objective research and analysis to inform investment decision-making.

Core Principles

Analytical Rigor: Apply systematic, data-driven analysis with appropriate skepticism. Validate information across multiple sources and flag any inconsistencies or data quality concerns.
Investment Focus: Frame all analysis through the lens of investment implications. Connect operational details to financial performance, competitive positioning, and risk/return profiles.
Objectivity: Present balanced analysis that acknowledges both positive and negative factors. Avoid promotional language and maintain professional neutrality throughout.
Timely Information: Attempt to present the latest information as of today, {date_today}. Note dates of sources referenced. Do not treat possibly outdated information as current.
Forward-looking perspective: Include historical information for context to understand the current situation and future prospects. Information that impacts future prospects and drives investment returns going forward is of most interest.
Research Methodology
Multi-Source Verification: Cross-reference information from company filings, financial databases, industry reports, and credible news sources. When sources conflict, note discrepancies and assess reliability.
Systematic Information Gathering: Use available tools methodically:

Financial data APIs for quantitative metrics and peer comparisons
Web search for recent developments, analyst coverage, and industry context
Company filings and investor materials for official positions
News aggregation for market sentiment and risk factor identification

Quality Standards: Prioritize authoritative sources (SEC filings, earnings calls, established financial media, recognized research firms) over speculative or promotional content. Flag when analysis relies on limited or potentially biased sources.
Analytical Framework
Strategic Context: Position the company within its industry ecosystem, considering competitive dynamics, market maturity, regulatory environment, and secular trends.
Financial Analysis: Go beyond basic metrics to understand business quality, cash generation, capital efficiency, and financial flexibility. Connect financial trends to underlying business drivers.
Risk Assessment: Identify and categorize risks systematically (operational, financial, competitive, regulatory, ESG). Assess probability and potential impact of key risk scenarios.
Valuation Perspective: While not providing specific price targets, discuss valuation approaches relevant to the business model and highlight key variables that drive investment returns.
Output Expectations
Professional Tone: Write in the clear, authoritative style of institutional research. Use precise financial terminology while remaining accessible to sophisticated investors.
Evidence-Based Conclusions: Support all analytical points with specific data, examples, or credible sources. Distinguish between factual information and analytical interpretation.
Actionable Insights: Focus on information that would influence investment decisions. Highlight critical factors for ongoing monitoring and key questions for further due diligence.
Structured Presentation: Follow the specific organizational structure provided in the user prompt, ensuring logical flow and appropriate depth for each section.
Key Reminders

Maintain objectivity and avoid advocacy for any investment position
Clearly distinguish between company-provided information and independent analysis
Note when information is incomplete, uncertain, or requires further investigation
Focus on material factors that could significantly impact investment outcomes
Present analysis that would meet institutional investment committee standards

Your goal is to produce a comprehensive, professional-grade company profile that enables informed investment decision-making through rigorous research and balanced analysis."""

In [ ]:
user_prompt = f"""

Using everything found so far and everything you can find by using tools and doing deep research, write a report on Tesla (symbol TSLA) in the straightforward factual style of a Wall Street equity research analyst, in 8 sections:

1. Profile
• History with origin story and key historical milestones
• Core business and competitors
• Major news events since {last_year}

2. Business Model:
• Describe their core businesses, products and services.
• Outline their key revenue streams, customer segments, and monetization strategies.
• Analyze key characteristics of markets it operates in:
    - customer acquisition costs
    - retention metrics
    - sales cycles
    - seasonal or cyclical business patterns
    - margins, market size, growth trajectory and factors affecting them
• Explain sources of competitive advantage such as network effects, switching costs, brands, intellectual property, regulatory moats, and other barriers to entry.

3. Competitive Landscape:
• Identify their main competitors, including direct, adjacent, and emerging competitors.
• Compare key metrics such as market share, product differentiation, pricing power, and growth trajectories.

4. Supply Chain Positioning:
• Describe their role in the upstream (supplier-side) and downstream (customer/distribution) parts of the supply chain.
• Identify key suppliers, partners, distributors, and any major dependencies or concentrations.

5. Financial and Operating Leverage:
• Analyze the company’s use of financial leverage (debt levels, interest obligations, credit ratings).
• Analyze operating leverage (fixed vs. variable cost structure, scalability, margin sensitivity to revenue changes).
• Analyze cash flow generation and working capital dynamics.
• Analyze capital allocation strategy (dividends, buybacks, reinvestment).

6. Valuation:
• Identify appropriate valuation methodologies, including income-based (e.g., DCF), asset-based (eg book value and sum of parts), market-based (e.g. peer multiples and comparisons), and LBO analysis
• Highlight important valuation inputs and metrics (growth rates, margins, discount rates, terminal value assumptions).
• Summarize current ratings and analyst opinions, including recent changes.
• Note the stock's volatility, liquidity, if it is widely covered and owned, if it is a hedge fund story stock or meme stock, what macro factors it is sensitive to

7. Recent developments, News Search and Risk Factors:
• Conduct a deep news search for significant positive and negative news items since {last_year}, including:
    • Revenue and earnings trends
    • Management changes
    • New product launches
    • Restructurings, mergers, acquisitions, divestitures, strategic partnerships
    • Short-seller reports or allegations.
    • Regulatory investigations or lawsuits.
    • Product failures, operational issues, or supply chain disruptions.
    • Major wins (e.g., partnerships, large customer wins, successful product launches).
    • Insider trading activity and institutional ownership changes
• Summarize key themes from media coverage, analyst reports, and public filings since {last_year}.
• Note controversies, reputational risks, and governance concerns if any.
• Discuss what companies might be potential acquisition targets or acquirers of the company, based on overlapping or complementary customer bases, product offerings, and technical capabilities.
• Note any other key themes or trends that you think are important.

8. Overall Assessment:
• Summarize the company’s strategic position and the stock's investment risk/reward profile
    • Strengths, weaknesses, opportunities, threats
    • Bear case and bull case
    • The level of risk
    • Any critical “watch points” for further due diligence and ongoing monitoring.

"""

In [ ]:
!ls tmp/tx0ybr2zo/ 


In [ ]:
balance sheet
income statement


In [ ]:
upload_files = {
    'edgar_10k_item1.txt': f'SEC 10K Item 1',
    'perplexity_profile.md': f'Company profile from Perplexity',
    'peers.txt': f'List of Peers',
    'fundamentals_av.json': f'Fundamentals for company and peers',
    'perplexity_ratings.md': f'Analyst ratings from Perplexity',
    'technicals.md': f'Current technicals',
    'perplexity_news.md': f'Company news from Perplexity',
    'wikipedia.md': f'Wikipedia profile'
}

In [ ]:
# current_files = [f"{temp_dir}/{symbol}_{f}" for f in upload_files]
current_files = [str(f) for f in Path(temp_dir).iterdir() if f.is_file()]

# Sort files by size (smallest to largest)
current_files_by_size = sorted(current_files, key=lambda f: Path(f).stat().st_size)
current_files_by_size


In [ ]:
for file_path in current_files_by_size:
    parts = file_path.split('_', 1)
    result = file_path[len(parts[0])+1:]
    if result in upload_files:
        print(result, ": ", upload_files[result])
    else:
        print("notfound") 

In [ ]:
import tiktoken

# Initialize tiktoken encoder (using GPT-4 encoding)
encoding = tiktoken.get_encoding("cl100k_base")  # or "gpt-4" 

additional_context = ""
total_tokens = 0
files_processed = []

for file_path in current_files_by_size:
    try:
        # Read file content
        with open(file_path, 'r', encoding='utf-8') as file:
            file_content = file.read()
        
        # Count tokens in this file
        file_tokens = len(encoding.encode(file_content))
        
        # Check if adding this file would exceed the limit
        if total_tokens + file_tokens > 90000:
            print(f"Stopping before {file_path} - would exceed 90,000 tokens")
            break
        
        # Add to context and update counters
        parts = file_path.split('_', 1)
        section_title = file_path[len(parts[0])+1:]
        additional_context += f"# {symbol} {section_title}\n"
        additional_context += file_content + "\n\n"  # Add some separation between files
        total_tokens += file_tokens
        files_processed.append(file_path)
        
        print(f"Added {file_path}: {file_tokens} tokens (Total: {total_tokens})")
        
    except (UnicodeDecodeError, OSError, FileNotFoundError) as e:
        print(f"Error reading {file_path}: {e}")
        continue

print(f"\nProcessed {len(files_processed)} files")
print(f"Total tokens: {total_tokens}")
print(f"Total characters in additional_context: {len(additional_context)}")


In [ ]:
print(additional_context)

In [ ]:
keylist = list(upload_files.keys())
keylist = keylist[len(files_processed):]
file_list=[client.files.create(
    file=open(f"{temp_dir}/{symbol}_{f}", "rb"), purpose="assistants") 
           for f in keylist]


In [ ]:
file_list

In [ ]:
    • company +  "analyst report" OR "research note" downgrade OR upgrade
    • use perplexity to search for company + "profile" or "executive profile" and ask: "What are the most significant investigative reports and executive profiles about company published in 2023-2024?"


In [ ]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

response = client.responses.create(
  model="o3-deep-research",
  input=[
    {
      "role": "developer",
      "content": [
        {
          "type": "input_text",
          "text": system_prompt,
        }
      ]
    },
    {
      "role": "user",
      "content": [
        {
          "type": "input_text",
          "text": user_prompt,
        }
      ]
    }
  ],
  reasoning={
    "summary": "auto"
  },
  tools=[
    {
      "type": "web_search_preview"
    },
    {
      "type": "code_interpreter",
      "container": {
        "type": "auto",
        "file_ids": [f.id for f in file_list]
      }
    }
  ]
)

In [ ]:
since current year -1

In [ ]:
response_text = response.output[-1].content[0].dict()['text']
response_text = response_text.replace(r'$',r'\$')
display(Markdown(response_text))
